# Text cleaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mary-Vadi/nlp-toolkit/blob/main/notebooks/01_text_cleaning.ipynb)

Walks through the individual cleaning functions, the three presets, and token-level preprocessing.

In [ ]:
import os, sys, subprocess

try:
    import nlp_toolkit
except ImportError:
    if os.path.isdir("../src/nlp_toolkit"):  # local clone
        sys.path.insert(0, os.path.abspath("../src"))
    else:  # Colab
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/Mary-Vadi/nlp-toolkit.git", "langdetect"], check=True)
    import nlp_toolkit

nlp_toolkit.__version__

## Sample input

In [ ]:
import pandas as pd
from nlp_toolkit import CleaningPipeline, cleaning as C

pd.set_option("display.max_colwidth", None)

raw = ("<p>Check THIS out!!!   https://example.com/page?id=7 &amp; email me@uni.ac.uk</p> "
       "\u201cIt\u2019s sooooo GOOD\u201d \u2014 @maryam #NLP \U0001F60D costs 3.50 in 2026")
print(raw)

## Individual functions

In [ ]:
examples = {
    "unescape_html": C.unescape_html(raw),
    "remove_html_tags": C.remove_html_tags(raw),
    "fix_smart_characters": C.fix_smart_characters(raw),
    "replace_urls": C.replace_urls(raw, token="<URL>"),
    "replace_emails": C.replace_emails(raw, token="<EMAIL>"),
    "replace_mentions": C.replace_mentions(raw),
    "handle_hashtags": C.handle_hashtags(raw),
    "remove_emojis": C.remove_emojis(raw),
    "expand_contractions": C.expand_contractions(raw),
    "reduce_repeated_characters": C.reduce_repeated_characters(raw),
    "replace_numbers": C.replace_numbers(raw, token="<NUM>"),
    "to_lowercase": C.to_lowercase(raw),
    "remove_punctuation": C.remove_punctuation(raw),
}
pd.DataFrame({"output": {k: C.normalize_whitespace(v) for k, v in examples.items()}})

## Step-by-step trace

`trace()` returns the text after each step, which is useful when a pipeline produces something unexpected.

In [ ]:
pipe = CleaningPipeline.from_preset("classic")
pd.DataFrame(pipe.trace(raw), columns=["step", "text"])

## Presets

`transformer` keeps case and punctuation, since pretrained tokenizers rely on them. `classic` normalises aggressively for sparse features. `social` masks URLs and user handles.

In [ ]:
pd.DataFrame({"output": {p: CleaningPipeline.from_preset(p)(raw) for p in ["transformer", "classic", "social"]}})

## Custom pipelines

In [ ]:
from functools import partial

custom = CleaningPipeline([
    C.unescape_html,
    C.remove_html_tags,
    C.fix_smart_characters,
    partial(C.replace_urls, token="<URL>"),
    partial(C.replace_numbers, token="<NUM>"),
    C.normalize_whitespace,
])
print(custom(raw))

# modify a preset
modified = CleaningPipeline.from_preset("transformer").add(C.remove_emojis).remove("replace_emails")
print(modified.describe())

## Token-level preprocessing

For TF-IDF / bag-of-words models only. Negations are kept when removing stopwords.

In [ ]:
from nlp_toolkit import preprocessing as P

tokens = P.simple_tokenize("The students were not studying the cleaned datasets carefully".lower())
no_stop = P.remove_stopwords(tokens)

pd.DataFrame({"result": {
    "tokens": tokens,
    "stopwords removed": no_stop,
    "stemmed": P.stem(no_stop),
    "lemmatized": P.lemmatize(no_stop),
    "bigrams": P.ngrams(no_stop, 2),
}})

## Batch cleaning

In [ ]:
CleaningPipeline.from_preset("classic").clean_many(
    ["Hello   WORLD!!", "<b>Bold</b> &amp; brave", None, "Visit www.salford.ac.uk"]
)